# Training of our own detector model (issue #46, work package 7)

Runs `training/model/train.py` in a GPU session on Kaggle or Colab. What the script does and how a run is continued: `training/model/README.md`.

Before the first run:
- **Dataset.** Upload `training/data/dataset/v1/train.jsonl`, `validation.jsonl` and `test.jsonl` as a **private** dataset (Kaggle: "New Dataset"; Colab: a folder in your Drive). Not every source may be redistributed, so the dataset must not be public.
- **GPU.** Kaggle: Settings → Accelerator → GPU, and Internet on. Colab: Runtime → Change runtime type → GPU.

One run per session. A session that ends before the run is done is continued by running the notebook again with `RESUME_FROM` set.

In [ ]:
RUN = "ettin-68m"        # a candidate of model/train.py --list
COMMIT = "main"          # branch or commit of the repository to train with; a commit for a result that is reported
HOURS = 11               # stop and save before the session ends (Kaggle: 12 h per session)
DATA = "/kaggle/input/aivsai-dataset-v1"   # folder with train.jsonl, validation.jsonl, test.jsonl
RESUME_FROM = ""         # folder with the checkpoint.pt of an earlier session of this run, or empty
EXTRA = ""               # further options of train.py, e.g. "--batch 16"

In [ ]:
import os, pathlib

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = pathlib.Path("/kaggle/working" if ON_KAGGLE else "/content")
if not ON_KAGGLE and DATA.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
OUT = WORK / "runs" / RUN   # Colab: set this to a folder in your Drive, /content is lost with the session

%cd $WORK
!rm -rf ai-check-extension && git clone -q https://github.com/OJPOJ/ai-check-extension.git && git -C ai-check-extension checkout -q $COMMIT && git -C ai-check-extension log -1 --oneline
!pip install -q "transformers>=5.17"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
resume = f"--resume-from {RESUME_FROM}" if RESUME_FROM else ""
%cd $WORK/ai-check-extension/training
!python model/train.py $RUN --data $DATA --out $OUT --hours $HOURS $resume $EXTRA

The folder of the run is the output of the session. To continue: `checkpoint.pt` (Kaggle: save the version, add its output as input of the next session and set `RESUME_FROM` to it). When the run is done, copy `run.json`, `log.jsonl`, `scores/` and `model.pt` to `training/data/runs/<RUN>/` of the repository; `checkpoint.pt` is no longer needed.

In [ ]:
!ls -la $OUT $OUT/scores 2>/dev/null
!cd $OUT && zip -q -r $WORK/$RUN-results.zip run.json log.jsonl scores 2>/dev/null; ls -la $WORK/*.zip